In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.3 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file 

In [3]:
import time

# Setup observability
USE_OBSERVABILITY = True
conversation_id = f"massi-uc1-run-{time.time()}"

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )

1. **We start with a partial metadata file:**

In [4]:
import json

# Adapted from <https://github.com/MoseleyBioinformaticsLab/miagis/blob/main/examples/Paper_2/base_metadata.json>
input_base_metadata = {
  "entry_version": 1,
  # "entry_id": "KY SOP Paper",
  # "description": "Data used for the paper titled \"A FAIR approach for detecting and sharing PFAS hot-spot areas and water systems\" DOI: ",
  "products": [
    "Combined map",
    "Water map",
    "Map Sewer",
    "SWAPP map Ky",
    "Hotspot Map",
    "PFAS_sampling_and_purchasing_and_intake_detail",
    "hot-spot_map",
    "New_KDEP_PFAS_Data_Compiled-2020_with_latlong"
  ]
}

with open("base_metadata.json", "w") as bmf:
    json.dump(input_base_metadata, bmf)

In [5]:
DEFAULT_ID = "KY SOP Paper"
DEFAULT_DESCRIPTION = "Data used for the paper titled \"A FAIR approach for detecting and sharing PFAS hot-spot areas and water systems\" DOI: "
DEFAULT_VERSION = "1"
DEFAULT_PRODUCTS = "Combined map, Water map, Map Sewer, SWAPP map Ky, Hotspot Map, PFAS_sampling_and_purchasing_and_intake_detail, hot-spot_map, New_KDEP_PFAS_Data_Compiled-2020_with_latlong"

def mock_query_replies(state):
    target_attribute = state.values["current_query_prompt"]["key"]
    target_path = state.values["current_query_prompt"]["json_path"]
    match target_attribute:
        case str(target_attribute) if ("entry_id" in target_attribute or "entry_id" in target_path):
            return DEFAULT_ID
        case str(target_attribute) if ("description" in target_attribute or "description" in target_path):
            return DEFAULT_DESCRIPTION
        case str(target_attribute) if ("entry_version" in target_attribute or "entry_version" in target_path):
            return DEFAULT_VERSION
        case str(target_attribute) if ("products" in target_attribute or "products" in target_path):
            return DEFAULT_PRODUCTS
        case _: 
            raise ValueError(f"unexpected attribute: target_attribute")

In [6]:
from unittest.mock import patch # to bypass the user input

IS_MOCK_RUN=True

2. **We then use the Metadata Assistant to complete the file** 

In [7]:
from massi.agents.massi import MassiScenarioFactory, Config

In [8]:
# config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "lfm2.5-thinking"})
config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
factory = MassiScenarioFactory(config)

In [9]:
graph = factory.base_metadata_graph()

In [10]:
import grandalf

print(graph.get_graph().draw_mermaid())
graph.get_graph().print_ascii()

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	ensure_metadata_file_presence(ensure_metadata_file_presence)
	select_working_base_metadata_file(select_working_base_metadata_file)
	check_metadata_completion(check_metadata_completion)
	interpret_metadata_checks(interpret_metadata_checks)
	plan_user_query(plan_user_query)
	pull_ready_user_query(pull_ready_user_query<hr/><small><em>__interrupt = after</em></small>)
	prompt_user_query(prompt_user_query)
	register_user_query_response(register_user_query_response<hr/><small><em>__interrupt = before,after</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> ensure_metadata_file_presence;
	check_metadata_completion --> interpret_metadata_checks;
	ensure_metadata_file_presence --> select_working_base_metadata_file;
	interpret_metadata_checks --> plan_user_query;
	plan_user_query --> pull_ready_user_query;
	prompt_user_query --> register_user_query_response;
	pull_ready_user_query --> pro

In [11]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [12]:
import pprint

In [13]:
from massi.runtimes.jupyter.flow import stream_until_iterruption

> **_Now we begin the interaction_**

In [14]:
# Run the graph until the first interruption
stream_until_iterruption(graph, {}, thread)


==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']

==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']
---- selected_base_metadata_file ----
'/home/jovyan/work/_demo/base_metadata.json'

==== new event ====
---- messages ----
[]
---- base_metadata_files ----
['/home/jovyan/work/_demo/base_metadata.json']
---- selected_base_metadata_file ----
'/home/jovyan/work/_demo/base_metadata.json'
---- schema_validation_errors ----
[   {   'message': "'entry_id' is a required property",
        'encountered_value': [   'entry_version',
                                 'entry_id',
                                 'description',
                                 'products'],
        'json_path': '$',
        'failed_key': 'required'},
    {   'message': "'description' is a required property",
        'encountered_value': [   'entry_version',
                        

> **_After the first interrupt -- first user query is ready_**

In [15]:
state = graph.get_state(thread)
pprint.pp(state.values.keys())
print("~~~~")
for key in state.values.keys():
    print(f"\n---- {key} ----")
    pprint.pp(state.values[key], indent=4)

dict_keys(['messages', 'base_metadata_files', 'selected_base_metadata_file', 'schema_validation_errors', 'errors_summary', 'query_plan', 'query_plan_completed', 'current_step_pointer', 'current_query_prompt', 'original_metadata'])
~~~~

---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}

In [16]:
pprint.pp(state.values["query_plan"], indent=4)
print("~~~~")
pprint.pp(state.values["query_plan_completed"], indent=4)
print("~~~~")
pprint.pp(state.values["current_step_pointer"], indent=4)
print("~~~~")
pprint.pp(state.values["current_query_prompt"], indent=4)

{   'queries': [   {   'id': 1,
                       'key': 'entry_id',
                       'json_path': '$',
                       'priority': 5,
                       'query': 'Please provide a unique identifier for this '
                                'data entry. This identifier should be clear '
                                'and follow any naming conventions that are '
                                'relevant to your project.'},
                   {   'id': 2,
                       'key': 'description',
                       'json_path': '$',
                       'priority': 5,
                       'query': 'Please provide a brief yet comprehensive '
                                'description of this data entry. Include '
                                "details about the data's purpose, its "
                                'relevance to GIS, and any specific '
                                'characteristics that are important for users '
                   

> **_After the first interrupt - will continue with prompting the user_**
>
> - make sure to enter a value

In [17]:
if IS_MOCK_RUN: 
    with patch('builtins.input', return_value=mock_query_replies(graph.get_state(thread))):
        stream_until_iterruption(graph, None, thread)
else: 
    stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'pr

> **_After second interrupt - user response given but not registed_**

In [18]:
print(graph.get_state(thread).values["current_query_response"])
print(graph.get_state(thread).values["selected_base_metadata_file"])

KY SOP Paper
/home/jovyan/work/_demo/base_metadata.json


> **_After the second interrupt -- metadata will now be updated_**

In [19]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'pr

In [20]:
graph.get_state(thread).values

{'messages': [AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'prompt_eval_duration': 903796145

In [21]:
keys = ["original_metadata", "updated_metadata"]
for key in keys:
    print(f"\n---- {key} ----")
    pprint.pp(graph.get_state(thread).values[key], indent=4)


---- original_metadata ----
{   'entry_version': 1,
    'products': [   'Combined map',
                    'Water map',
                    'Map Sewer',
                    'SWAPP map Ky',
                    'Hotspot Map',
                    'PFAS_sampling_and_purchasing_and_intake_detail',
                    'hot-spot_map',
                    'New_KDEP_PFAS_Data_Compiled-2020_with_latlong']}

---- updated_metadata ----
{   'metadata': {   'entry_version': 1,
                    'products': [   'Combined map',
                                    'Water map',
                                    'Map Sewer',
                                    'SWAPP map Ky',
                                    'Hotspot Map',
                                    'PFAS_sampling_and_purchasing_and_intake_detail',
                                    'hot-spot_map',
                                    'New_KDEP_PFAS_Data_Compiled-2020_with_latlong'],
                    'entry_id': 'KY SOP Paper',
     

> **_Now we can check the result of the metadata update_**

In [22]:
import json
with open(graph.get_state(thread).values["selected_base_metadata_file"], "r") as bmf:
    base_metadata = json.load(bmf)
    print(base_metadata)

{'entry_version': 1, 'products': ['Combined map', 'Water map', 'Map Sewer', 'SWAPP map Ky', 'Hotspot Map', 'PFAS_sampling_and_purchasing_and_intake_detail', 'hot-spot_map', 'New_KDEP_PFAS_Data_Compiled-2020_with_latlong'], 'entry_id': 'KY SOP Paper', 'description': 'KY SOP Paper'}


> **_After the second interrupt - will continue to the next query step (because two values are missing)_**

In [23]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'pr

In [24]:
graph.get_state(thread).values["current_query_prompt"]

{'id': 2,
 'key': 'description',
 'json_path': '$',
 'priority': 5,
 'query': "Please provide a brief yet comprehensive description of this data entry. Include details about the data's purpose, its relevance to GIS, and any specific characteristics that are important for users of GIS applications."}

> **_After the third interrupt - will prompt the next query step (because two values are missing)_**
>
> - don't forget to provide an input

In [25]:
if IS_MOCK_RUN: 
    with patch('builtins.input', return_value=mock_query_replies(graph.get_state(thread))):
        stream_until_iterruption(graph, None, thread)
else: 
    stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'pr

In [26]:
print(f"{graph.get_state(thread).values["query_plan_completed"]=}")

graph.get_state(thread).values["query_plan_completed"]=True


> **_After the four interrupt -- metadata will now be updated_**

In [27]:
stream_until_iterruption(graph, None, thread)


==== new event ====
---- messages ----
[   AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'pr

In [28]:
graph.get_state(thread).values

{'messages': [AIMessage(content='Here are the JSON schema validation errors explained in a human-readable format:\n\n- The required property "entry_id" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\n- The required property "description" is missing from the JSON data.\n  - Encountered values: "entry_version", "entry_id", "description", "products"\n  - JSON path: $\n  - Failed key: required\n\nIn summary, the JSON data provided for the MIAGIS base metadata file is missing the required properties "entry_id" and "description". These properties are necessary for the data to be considered valid against the JSON schema.', additional_kwargs={}, response_metadata={'model': 'granite4:3b', 'created_at': '2026-03-13T15:09:28.148598Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353641913, 'load_duration': 4491358492, 'prompt_eval_count': 180, 'prompt_eval_duration': 903796145

In [29]:
keys = ["original_metadata", "updated_metadata"]
for key in keys:
    print(f"\n---- {key} ----")
    pprint.pp(graph.get_state(thread).values[key], indent=4)


---- original_metadata ----
{   'entry_version': 1,
    'products': [   'Combined map',
                    'Water map',
                    'Map Sewer',
                    'SWAPP map Ky',
                    'Hotspot Map',
                    'PFAS_sampling_and_purchasing_and_intake_detail',
                    'hot-spot_map',
                    'New_KDEP_PFAS_Data_Compiled-2020_with_latlong']}

---- updated_metadata ----
{   'metadata': {   'entry_version': 1,
                    'products': [   'Combined map',
                                    'Water map',
                                    'Map Sewer',
                                    'SWAPP map Ky',
                                    'Hotspot Map',
                                    'PFAS_sampling_and_purchasing_and_intake_detail',
                                    'hot-spot_map',
                                    'New_KDEP_PFAS_Data_Compiled-2020_with_latlong'],
                    'entry_id': 'KY SOP Paper',
     

> **_Now we can check the result of the metadata update_**

In [30]:
import json
with open(graph.get_state(thread).values["selected_base_metadata_file"], "r") as bmf:
    base_metadata = json.load(bmf)
    pprint.pp(base_metadata, indent=4)

{   'entry_version': 1,
    'products': [   'Combined map',
                    'Water map',
                    'Map Sewer',
                    'SWAPP map Ky',
                    'Hotspot Map',
                    'PFAS_sampling_and_purchasing_and_intake_detail',
                    'hot-spot_map',
                    'New_KDEP_PFAS_Data_Compiled-2020_with_latlong'],
    'entry_id': 'KY SOP Paper',
    'description': "Data used for the paper titled 'A FAIR approach for "
                   'detecting and sharing PFAS hot-spot areas and water '
                   "systems' DOI: 2023. This metadata entry provides "
                   'geospatial data relevant to identifying and visualizing '
                   'PFAS contamination hotspots and associated water systems, '
                   'supporting environmental monitoring and decision-making '
                   'for water quality management.'}
